In [ ]:
import os
import torch
from torch.utils.data import DataLoader, TensorDataset
from transformers import DeepseekV3Config, DeepseekV3ForCausalLM
import numpy as np
from tqdm import tqdm
from torch.nn.parallel import DataParallel

In [ ]:
train_dir = "/kaggle/input/token-splits/train/train"
val_dir   = "/kaggle/input/token-splits/val/val"

In [ ]:
seq_len = 1024
batch_size = 4
num_epochs = 1
learning_rate = 5e-5
checkpoint_interval = 5000  # batches

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_gpus = torch.cuda.device_count()
print(f"Available GPUs: {num_gpus}")
print(f"Using device: {device}")

In [ ]:
configuration = DeepseekV3Config(
    vocab_size=64000,
    hidden_size=480,
    intermediate_size=1920,
    num_hidden_layers=8,
    num_attention_heads=8,
    num_key_value_heads=8,
    n_routed_experts=None,
    num_experts_per_tok=None,
    n_shared_experts=None,
    moe_layer_freq=0,
    first_k_dense_replace=20
)

In [ ]:
model = DeepseekV3ForCausalLM(configuration)

# Enable multi-GPU training
if num_gpus > 1:
    print(f" Using {num_gpus} GPUs with DataParallel!")
    model = DataParallel(model)
else:
    print("Using single GPU")

model.to(device)
model.train()

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
print(f"Model parameters: {model.module.num_parameters() / 1e6:.2f}M" if num_gpus > 1 else f"Model parameters: {model.num_parameters() / 1e6:.2f}M")


In [ ]:
train_files = sorted([os.path.join(train_dir, f) for f in os.listdir(train_dir) if f.endswith(".npy")])
val_files   = sorted([os.path.join(val_dir, f) for f in os.listdir(val_dir) if f.endswith(".npy")])


In [ ]:
# Load checkpoint if exists
checkpoint_path = "/kaggle/input/checkpotfile31/checkpoint (8).pt"
checkpoint = torch.load(checkpoint_path, map_location=device)

model_to_load = model.module if num_gpus > 1 else model
model_to_load.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

start_epoch = checkpoint.get('epoch', 0)
start_file_idx = 61
start_batch_idx = 0

# Restore loss accumulators if they exist
epoch_loss_total = checkpoint.get('epoch_loss_total', 0)
epoch_batches = checkpoint.get('epoch_batches', 0)

print(f"Resuming from epoch {start_epoch}, file {start_file_idx}, batch {start_batch_idx}")
print(f"Resuming loss totals: {epoch_loss_total:.4f} over {epoch_batches} batches")


In [ ]:
# ------------------------------
# Training loop across all files
# ------------------------------
for epoch in range(num_epochs):
    print(f"\n===== Epoch {epoch+1} =====")
    if epoch != start_epoch:
        epoch_loss_total = 0
        epoch_batches = 0

    for file_idx, train_file in enumerate(train_files):
        if epoch == start_epoch and file_idx < start_file_idx:
            continue
        print(f"Loading {train_file} ...")
        tokens = np.load(train_file)
        tokens = torch.tensor(tokens, dtype=torch.long)

        num_sequences = len(tokens) // seq_len
        tokens = tokens[:num_sequences * seq_len].reshape((num_sequences, seq_len))
        dataset = TensorDataset(tokens)
        dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

        progress_bar = tqdm(dataloader, desc=f"Epoch {epoch+1}, file {file_idx+1}")
        for batch_idx, batch in enumerate(progress_bar):
            if epoch == start_epoch and file_idx == start_file_idx and batch_idx <= start_batch_idx:
                continue
            input_ids = batch[0].to(device)
            outputs = model(input_ids, labels=input_ids)

            if num_gpus > 1:
                loss = outputs.loss.mean()
            else:
                loss = outputs.loss

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # Track loss globally
            epoch_loss_total += loss.item()
            epoch_batches += 1
            global_avg_loss = epoch_loss_total / epoch_batches
            progress_bar.set_postfix({'globall_avg_loss': f"{loss:.4f}"})

            # Optional batch checkpoint
            if (batch_idx + 1) % checkpoint_interval == 0:
                checkpoint_path = f"./checkpoint_epoch{epoch+1}_file{file_idx+1}_batch{batch_idx+1}"
                os.makedirs(checkpoint_path, exist_ok=True)
                model_to_save = model.module if num_gpus > 1 else model
                model_to_save.save_pretrained(checkpoint_path)
                torch.save({
                    'model_state_dict': model_to_save.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'epoch': epoch,
                    'file_idx': file_idx,
                    'batch_idx': batch_idx,
                    'epoch_loss_total': epoch_loss_total,
                    'epoch_batches': epoch_batches
                }, f"{checkpoint_path}/checkpoint.pt")
                print(f"Checkpoint saved at batch {batch_idx+1}")

        del tokens, dataset, dataloader
        torch.cuda.empty_cache()

    # Compute global average loss for the epoch
    epoch_avg_loss = epoch_loss_total / epoch_batches
    print(f"Epoch {epoch+1} training loss: {epoch_avg_loss:.4f}")

    # Epoch checkpoint
    checkpoint_path = f"./checkpoint_epoch{epoch+1}"
    os.makedirs(checkpoint_path, exist_ok=True)
    model_to_save = model.module if num_gpus > 1 else model
    model_to_save.save_pretrained(checkpoint_path)
    torch.save({
        'model_state_dict': model_to_save.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'epoch': epoch
    }, f"{checkpoint_path}/checkpoint.pt")
    print(f"Checkpoint saved after epoch {epoch+1}: {checkpoint_path}")

    # ------------------------------
    # VALIDATION
    # ------------------------------
    model.eval()
    val_loss_total = 0
    total_val_batches = 0

    with torch.no_grad():
        for val_file in val_files:
            tokens = np.load(val_file)
            tokens = torch.tensor(tokens, dtype=torch.long)
            num_sequences = len(tokens) // seq_len
            tokens = tokens[:num_sequences * seq_len].reshape((num_sequences, seq_len))
            dataset = TensorDataset(tokens)
            dataloader = DataLoader(dataset, batch_size=batch_size)

            for batch in dataloader:
                input_ids = batch[0].to(device)
                outputs = model(input_ids, labels=input_ids)
                if num_gpus > 1:
                    val_loss_total += outputs.loss.mean().item()
                else:
                    val_loss_total += outputs.loss.item()
                total_val_batches += 1

            del tokens, dataset, dataloader
            torch.cuda.empty_cache()

    avg_val_loss = val_loss_total / total_val_batches
    print(f"Epoch {epoch+1} validation loss: {avg_val_loss:.4f}")
    model.train()


In [ ]:
# ------------------------------
# FINAL MODEL SAVE
# ------------------------------
os.makedirs("./slm_multi", exist_ok=True)

# Save the actual model (not the DataParallel wrapper)
if num_gpus > 1:
    model.module.save_pretrained("./slm_multi")
else:
    model.save_pretrained("./slm_multi")
    
torch.save({
    'model_state_dict': model.module.state_dict() if num_gpus > 1 else model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'epoch': num_epochs,
}, "./slm_multi/final_checkpoint.pt")

print("Training complete. Model saved!")

In [ ]:
import torch
from transformers import PreTrainedTokenizerFast
# from your_model_file import DeepseekV3ForCausalLM  # import your class

# Load tokenizer from Kaggle input
tokenizer = PreTrainedTokenizerFast(tokenizer_file="/kaggle/input/final-cpt/multilingual_bpe_64k.json")
# Make sure special tokens are registered
tokenizer.pad_token = "<pad>"
tokenizer.unk_token = "<unk>"
tokenizer.bos_token = "<s>"
tokenizer.eos_token = "</s>"
model = DeepseekV3ForCausalLM(configuration)

checkpoint = torch.load("/kaggle/input/final-cpt/final_model.pt", map_location="cuda")
model.load_state_dict(checkpoint["model_state_dict"])


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

prompt = "Did yoy know that lightning"
inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=True)
inputs.pop("token_type_ids", None)
inputs = {k: v.to(device) for k, v in inputs.items()}
eos_id = tokenizer.eos_token_id  # should be 3 in your case

with torch.no_grad():
    outputs = model.generate(
    **inputs,
    max_length=100,
    eos_token_id=eos_id
)

print(f'Prompt:{prompt}')
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

# prompt = "Dead sea was"